# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duaaanadeem/ML-workspace/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule

I will rank pages higher when they have meaningful search volume and a lower-than-expected CTR for their search position.

The score will combine search volume with the CTR opportunity. Pages with higher opportunity will be placed higher in the action queue.

### Reason code

CTR_OPPORTUNITY — the page has enough search volume and its CTR suggests there may be an opportunity for improvement.

### Action label

REVIEW_CTR — review the page for a possible CTR improvement.


In [2]:
import duckdb

con = duckdb.connect()
print("DuckDB connection ready")

DuckDB connection ready


In [7]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [16]:
import os
import duckdb
from google.colab import userdata

# 1. Get your HF token
HF_TOKEN = userdata.get("HF_Token")

print("Token loaded:", HF_TOKEN is not None)

# 2. Create DuckDB connection
con = duckdb.connect()

# 3. Load HTTPFS extension
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# 4. Give DuckDB the Hugging Face token
con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face authentication ready.")

# 5. Test March 2026 data
query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

sample = con.execute(query).fetchdf()

print("Columns:")
print(sample.columns.tolist())

display(sample)

Token loaded: True
Hugging Face authentication ready.
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [17]:
# Show all column names clearly

for i, col in enumerate(sample.columns):
    print(i, col)

0 report_date
1 client_hash_id
2 content_hash_id
3 client_has_gsc
4 client_has_ga4
5 gsc_data_available
6 ga4_data_available
7 gsc_impressions
8 gsc_clicks
9 gsc_sum_position
10 gsc_avg_position
11 ga4_pageviews
12 ga4_sessions
13 ga4_users
14 ga4_engaged_sessions
15 ga4_total_engagement_sec
16 sessions_organic
17 sessions_direct
18 sessions_referral
19 sessions_social
20 sessions_paid
21 sessions_ai
22 ai_chatgpt
23 ai_perplexity
24 ai_gemini
25 ai_copilot
26 ai_claude
27 ai_meta
28 ai_other
29 scroll_events
30 month


In [19]:
import duckdb

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f'{REL}/fact_content_daily_performance'

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("FEB ready")
print("MAR ready")

FEB ready
MAR ready


In [21]:
query = f"""
SELECT *
FROM {FEB}
LIMIT 5
"""

sample = con.execute(query).fetchdf()

for i, col in enumerate(sample.columns):
    print(i, col)

0 report_date
1 client_hash_id
2 content_hash_id
3 client_has_gsc
4 client_has_ga4
5 gsc_data_available
6 ga4_data_available
7 gsc_impressions
8 gsc_clicks
9 gsc_sum_position
10 gsc_avg_position
11 ga4_pageviews
12 ga4_sessions
13 ga4_users
14 ga4_engaged_sessions
15 ga4_total_engagement_sec
16 sessions_organic
17 sessions_direct
18 sessions_referral
19 sessions_social
20 sessions_paid
21 sessions_ai
22 ai_chatgpt
23 ai_perplexity
24 ai_gemini
25 ai_copilot
26 ai_claude
27 ai_meta
28 ai_other
29 scroll_events
30 month


In [22]:
# Find columns related to CTR, position, impressions and clicks

cols = sample.columns.tolist()

for col in cols:
    if any(word in col.lower() for word in ["ctr", "position", "impression", "click"]):
        print(col)

gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position


In [23]:
# Signal 1: CTR vs Position

signal1 = con.execute(f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,

    SUM(gsc_clicks) AS clicks,
    SUM(gsc_impressions) AS impressions,

    ROUND(
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0),
        2
    ) AS ctr_percent

FROM {FEB}
WHERE gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL

GROUP BY position_bucket

ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21+' THEN 4
    END
""").fetchdf()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,clicks,impressions,ctr_percent
0,1-3,635987,159517.0,41416842.0,0.39
1,4-10,1121355,334039.0,97451993.0,0.34
2,11-20,420091,59791.0,19616648.0,0.30
3,21+,444349,32858.0,21643428.0,0.15


In [24]:
# Find columns related to freshness / staleness / refresh

cols = con.execute(f"""
    SELECT * FROM {FEB} LIMIT 1
""").fetchdf().columns.tolist()

for col in cols:
    if any(word in col.lower() for word in [
        "stale", "fresh", "refresh", "age", "days"
    ]):
        print(col)

ga4_pageviews
ga4_engaged_sessions
ga4_total_engagement_sec


In [25]:
# Signal 2: Search volume / impressions

signal2 = con.execute(f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN '<100'
        WHEN gsc_impressions < 1000 THEN '100-999'
        WHEN gsc_impressions < 10000 THEN '1K-9.9K'
        ELSE '10K+'
    END AS volume_bucket,

    COUNT(*) AS n,

    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,

    ROUND(
        100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0),
        2
    ) AS ctr_percent

FROM {FEB}
WHERE gsc_impressions IS NOT NULL
  AND gsc_impressions >= 0

GROUP BY volume_bucket

ORDER BY
    CASE volume_bucket
        WHEN '<100' THEN 1
        WHEN '100-999' THEN 2
        WHEN '1K-9.9K' THEN 3
        WHEN '10K+' THEN 4
    END
""").fetchdf()

signal2

,volume_bucket,n,total_impressions,total_clicks,ctr_percent
0,<100,6833451,44462502.0,126223.0,0.28
1,100-999,412247,105055037.0,350176.0,0.33
2,1K-9.9K,17104,29590462.0,109601.0,0.37
3,10K+,50,1020921.0,205.0,0.02


In [26]:
# Build the ranked baseline action queue

baseline = con.execute(f"""
WITH content_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(gsc_avg_position) AS avg_position

    FROM {FEB}

    WHERE gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL

    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        *,

        100.0 * clicks / NULLIF(impressions, 0) AS actual_ctr,

        CASE
            WHEN avg_position <= 3 THEN 0.0039
            WHEN avg_position <= 10 THEN 0.0034
            WHEN avg_position <= 20 THEN 0.0030
            ELSE 0.0015
        END AS expected_ctr

    FROM content_features
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    ROUND(avg_position, 2) AS avg_position,

    ROUND(actual_ctr, 4) AS actual_ctr,
    ROUND(expected_ctr, 4) AS expected_ctr,

    ROUND(
        impressions * GREATEST(expected_ctr - actual_ctr, 0),
        2
    ) AS score,

    'CTR_OPPORTUNITY' AS reason_code,
    'REVIEW_CTR' AS action

FROM scored

ORDER BY score DESC
""").fetchdf()

baseline.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,avg_position,actual_ctr,expected_ctr,score,reason_code,action
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,2.49,0.0005,0.0039,663.03,CTR_OPPORTUNITY,REVIEW_CTR
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,3.84,0.0000,0.0034,659.44,CTR_OPPORTUNITY,REVIEW_CTR
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,4.97,0.0010,0.0034,491.56,CTR_OPPORTUNITY,REVIEW_CTR
3,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,9.37,0.0000,0.0034,425.12,CTR_OPPORTUNITY,REVIEW_CTR
4,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,18.68,0.0000,0.0030,102.88,CTR_OPPORTUNITY,REVIEW_CTR
5,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,18472.0,0.0,0.66,0.0000,0.0039,72.04,CTR_OPPORTUNITY,REVIEW_CTR
6,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,15238.0,0.0,0.50,0.0000,0.0039,59.43,CTR_OPPORTUNITY,REVIEW_CTR
7,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,34.95,0.0000,0.0015,49.20,CTR_OPPORTUNITY,REVIEW_CTR
8,client_23a62021009f63c4,content_66d3e7a515e4ec68,12026.0,0.0,2.46,0.0000,0.0039,46.90,CTR_OPPORTUNITY,REVIEW_CTR
9,client_73cda7b4e4f265ea,content_4b3ab5ebb70090f1,10622.0,0.0,2.31,0.0000,0.0039,41.43,CTR_OPPORTUNITY,REVIEW_CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
import os

# Create output directory
os.makedirs("work/outputs", exist_ok=True)

# Save ranked queue
output_path = "work/outputs/baseline_action_score.csv"
baseline.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(baseline))
print("Columns:", list(baseline.columns))

Saved: work/outputs/baseline_action_score.csv
Rows: 153559
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'actual_ctr', 'expected_ctr', 'score', 'reason_code', 'action']


In [28]:
# Top 10 ranked opportunities

top10 = baseline.head(10).copy()

top10[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "avg_position",
        "actual_ctr",
        "expected_ctr",
        "score",
        "reason_code",
        "action"
    ]
]

,client_hash_id,content_hash_id,impressions,clicks,avg_position,actual_ctr,expected_ctr,score,reason_code,action
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,2.49,0.0005,0.0039,663.03,CTR_OPPORTUNITY,REVIEW_CTR
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,3.84,0.0000,0.0034,659.44,CTR_OPPORTUNITY,REVIEW_CTR
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,4.97,0.0010,0.0034,491.56,CTR_OPPORTUNITY,REVIEW_CTR
3,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,9.37,0.0000,0.0034,425.12,CTR_OPPORTUNITY,REVIEW_CTR
4,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,18.68,0.0000,0.0030,102.88,CTR_OPPORTUNITY,REVIEW_CTR
5,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,18472.0,0.0,0.66,0.0000,0.0039,72.04,CTR_OPPORTUNITY,REVIEW_CTR
6,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,15238.0,0.0,0.50,0.0000,0.0039,59.43,CTR_OPPORTUNITY,REVIEW_CTR
7,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,34.95,0.0000,0.0015,49.20,CTR_OPPORTUNITY,REVIEW_CTR
8,client_23a62021009f63c4,content_66d3e7a515e4ec68,12026.0,0.0,2.46,0.0000,0.0039,46.90,CTR_OPPORTUNITY,REVIEW_CTR
9,client_73cda7b4e4f265ea,content_4b3ab5ebb70090f1,10622.0,0.0,2.31,0.0000,0.0039,41.43,CTR_OPPORTUNITY,REVIEW_CTR


## 3. Top-10 Review

The top-ranked pages are prioritized because they have the largest estimated CTR opportunity based on search impressions, observed CTR, and expected CTR for their average search position.

For each page, the review records the proposed action, why the page was ranked, and what evidence could make the recommendation wrong.

| Rank | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | REVIEW_CTR | Very high impressions and CTR far below the expected CTR for position 2.49. | Query mix or SERP features may explain the low CTR rather than a page-level CTR problem. |
| 2 | REVIEW_CTR | Very high impressions, position 3.84, and zero observed clicks create a large CTR opportunity score. | The zero-click observation could reflect data quality or unusual query-level behavior. |
| 3 | REVIEW_CTR | High impressions at position 4.97 with actual CTR well below the expected CTR. | The average position may hide variation across individual queries. |
| 4 | REVIEW_CTR | High impressions at position 9.37 with zero clicks produce a large estimated CTR opportunity. | Query mix, SERP features, or incomplete click data could explain the result. |
| 5 | REVIEW_CTR | Strong impression volume and zero clicks at position 18.68 create a measurable CTR gap. | Lower search visibility at this position may limit the usefulness of the expected CTR benchmark. |
| 6 | REVIEW_CTR | Meaningful impressions with zero clicks and an average position of 0.66 indicate a large CTR gap. | The position or click data may contain aggregation or tracking issues. |
| 7 | REVIEW_CTR | Meaningful impressions and zero clicks at position 0.50 result in a high CTR opportunity score. | Extremely strong reported position with zero clicks should be validated against the underlying data. |
| 8 | REVIEW_CTR | High impressions with zero clicks and position 34.95 create an estimated CTR opportunity. | Very low search position may make the expected CTR assumption unreliable. |
| 9 | REVIEW_CTR | Meaningful impressions at position 2.46 with zero clicks create a substantial CTR gap. | Query-level SERP features or data quality could explain the unusually low CTR. |
| 10 | REVIEW_CTR | Meaningful impressions at position 2.31 with zero clicks produce a CTR opportunity score. | The recommendation could be wrong if the underlying click or position data is incomplete. |

In [29]:
# Week 4 self-check

print("Signal 1 table rows:", len(signal1))
print("Signal 2 table rows:", len(signal2))
print("Baseline rows:", len(baseline))
print("CSV exists:", os.path.exists("work/outputs/baseline_action_score.csv"))

print("\nReason codes:")
print(baseline["reason_code"].unique())

print("\nActions:")
print(baseline["action"].unique())

print("\nTop 10 rows:", len(baseline.head(10)))

Signal 1 table rows: 4
Signal 2 table rows: 4
Baseline rows: 153559
CSV exists: True

Reason codes:
['CTR_OPPORTUNITY']

Actions:
['REVIEW_CTR']

Top 10 rows: 10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [30]:
# Build the Top-20 review table

top20 = baseline.head(20).copy()

top20_review = top20[
    [
        "client_hash_id",
        "content_hash_id",
        "score",
        "impressions",
        "avg_position",
        "actual_ctr",
        "expected_ctr",
        "reason_code",
        "action"
    ]
].copy()

top20_review["confidence_note"] = top20_review.apply(
    lambda row:
        f"High score driven by {row['impressions']:.0f} impressions and a "
        f"{row['expected_ctr'] - row['actual_ctr']:.4f} CTR gap.",
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    lambda row:
        "Query mix, SERP features, or incomplete click/position data could "
        "explain the observed CTR gap.",
    axis=1
)

top20_review.insert(
    0,
    "rank",
    range(1, len(top20_review) + 1)
)

top20_review

,rank,client_hash_id,content_hash_id,score,impressions,avg_position,actual_ctr,expected_ctr,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,663.03,195648.0,2.49,0.0005,0.0039,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 195648 impressions and a ...,"Query mix, SERP features, or incomplete click/..."
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,659.44,193954.0,3.84,0.0000,0.0034,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 193954 impressions and a ...,"Query mix, SERP features, or incomplete click/..."
2,3,client_73cda7b4e4f265ea,content_8e1334d6356668e3,491.56,203401.0,4.97,0.0010,0.0034,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 203401 impressions and a ...,"Query mix, SERP features, or incomplete click/..."
3,4,client_73cda7b4e4f265ea,content_c9f840183215651b,425.12,125035.0,9.37,0.0000,0.0034,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 125035 impressions and a ...,"Query mix, SERP features, or incomplete click/..."
4,5,client_23a62021009f63c4,content_2f09787bdf392b16,102.88,34293.0,18.68,0.0000,0.0030,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 34293 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."
5,6,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,72.04,18472.0,0.66,0.0000,0.0039,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 18472 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."
6,7,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,59.43,15238.0,0.50,0.0000,0.0039,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 15238 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."
7,8,client_23a62021009f63c4,content_559cdd76da9306de,49.20,32799.0,34.95,0.0000,0.0015,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 32799 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."
8,9,client_23a62021009f63c4,content_66d3e7a515e4ec68,46.90,12026.0,2.46,0.0000,0.0039,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 12026 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."
9,10,client_73cda7b4e4f265ea,content_4b3ab5ebb70090f1,41.43,10622.0,2.31,0.0000,0.0039,CTR_OPPORTUNITY,REVIEW_CTR,High score driven by 10622 impressions and a 0...,"Query mix, SERP features, or incomplete click/..."


### Top-20 review conclusion

The baseline queue prioritizes pages with the largest estimated CTR opportunity using February feature-window data only.

The CTR-versus-position signal was **CONFIRMED** because CTR decreased consistently across the position buckets. The search-volume signal was **MIXED** because CTR did not increase consistently across all volume buckets, with the 10K+ bucket also having a small sample size.

The queue uses one action (`REVIEW_CTR`) and one reason code (`CTR_OPPORTUNITY`). The recommendations should be validated against query mix, SERP features, and the quality of the underlying GSC click and position data before being treated as confirmed opportunities.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [31]:
# Weak picks + leakage check

print("Baseline columns used for scoring:")
print(list(baseline.columns))

print("\nFeature window:")
print("February 2026 only")

print("\nFuture-window check:")
print("March 2026 data was not used to calculate the baseline score.")

# Check for suspicious product / outcome / future-related columns
suspicious = [
    col for col in baseline.columns
    if any(word in col.lower() for word in [
        "product", "flag", "label", "outcome", "churn",
        "future", "march"
    ])
]

print("\nSuspicious columns in baseline:")
print(suspicious if suspicious else "None")

print("\nLeakage verdict: PASS")

Baseline columns used for scoring:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'actual_ctr', 'expected_ctr', 'score', 'reason_code', 'action']

Feature window:
February 2026 only

Future-window check:
March 2026 data was not used to calculate the baseline score.

Suspicious columns in baseline:
None

Leakage verdict: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [32]:
# Final Week 4 self-check

checks = {
    "Signal 1 bucket table created": len(signal1) == 4,
    "Signal 2 bucket table created": len(signal2) == 4,
    "Baseline queue created": len(baseline) > 0,
    "CSV created": os.path.exists("work/outputs/baseline_action_score.csv"),
    "One reason code": baseline["reason_code"].nunique() == 1,
    "One action": baseline["action"].nunique() == 1,
    "Top-20 review created": len(top20_review) == 20,
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check}")

print("\nOverall:", "PASS" if all(checks.values()) else "CHECK FAILED")

PASS - Signal 1 bucket table created
PASS - Signal 2 bucket table created
PASS - Baseline queue created
PASS - CSV created
PASS - One reason code
PASS - One action
PASS - Top-20 review created

Overall: PASS
